# Annotation in Scanpy

In [1]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/SCWF00021/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *

### Load data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters_v3.h5ad')
adata.shape

### Dimensions

In [ ]:
# Dimensions
logger.info(f"Object dimensions (cells x genes): {adata.shape}")   

### Assign cell types

In [ ]:
logger.info("Setting cluster IDs ...")
# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.2'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.2', 'cell_type']].head())

### Annotations

In [ ]:
# Plot feature plots res harmony 1.0
logger.info("Plotting UMAP and feature plots palette A ...")
tbl_dir = "../results/13MANUSCRIPT_PLOTS_TABLES/plots/"

fig = plot_celltype_and_gene_features(
    adata=adata,
    final_genes=final_genes,
    umap_palette=custom_palette,
    cell_type_column='cell_type',
    ncols=3,
    figsize_width=10.5,
    base_height_per_row=1.0
)
plt.show()

fig.tight_layout()

fig.savefig(
    tbl_dir + "umap_and feature_plt.pdf",
    dpi=300,
    bbox_inches="tight"
)

# Save as SVG
fig.savefig(
    tbl_dir + "umap_and_feature_plt.svg",
    bbox_inches="tight" 
)

plt.close(fig)

### Cell counts

In [ ]:
(
    adata.obs['cell_type']
    .value_counts()
    .reset_index()
    .rename(columns={"index": "cell_type"})
    .to_csv(report_dir + "scanpy_L1_cell_type_counts.tsv", sep="\t", index=False)
)
adata.obs['cell_type'].value_counts()

### Key Metrics

In [ ]:
# Total number of nuclei (assuming one per cell)
logger.info(f"Total number of nuclei: {adata.n_obs}")

# Median number of nuclei per sample
logger.info(f"Median nuclei per sample: {adata.obs['sample'].value_counts().median()}")

# Median number of reads per cell (assuming adata.X is count matrix; handles sparse/dense)
counts = adata.layers['counts']
reads_per_cell = counts.sum(axis=1)
reads_per_cell = reads_per_cell.A1 if hasattr(reads_per_cell, 'A1') else reads_per_cell
median_reads = np.median(reads_per_cell)

logger.info(f"Median reads per cell: {median_reads}")

logger.info(f"Are nuclei from all 134 samples represented in all L1 cell pops? {all(adata.obs.groupby('cell_type')['sample'].nunique() == 134)}")

# Count how many of the 134 samples have >= 20 cells across all 7 major cell types
samples_passing = (adata.obs.groupby('sample')['cell_type'].value_counts().unstack(fill_value=0) >= 20).all(axis=1).sum()
logger.info(f"Number of samples with >= 20 cells in all 7 major types: {samples_passing}")

# Summary table of samples with >= 20 nuclei per cell type
sample_counts_per_type = (adata.obs.groupby('sample')['cell_type'].value_counts().unstack(fill_value=0) >= 20).sum().to_frame(name='samples_with_20_plus')
print(sample_counts_per_type)

### Percentage of cells per sample per cluster

In [ ]:
def plot_stacked_figure(adata, sample_column, color_column=None, barplot=True, boxplots=False, recalculate_columns=True):
    """
    Create a stacked figure. Supports boxplots for QC and/or a stacked barplot for proportions.
    Returns the figure object for saving.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np
    from scipy.sparse import issparse

    # 1. Data Processing & Recalculation logic
    adata.obs[sample_column] = adata.obs[sample_column].astype(str)
    
    if recalculate_columns:
        # Simple check for raw counts to update n_counts/n_genes
        first_values = adata.X[:5, :5].toarray() if issparse(adata.X) else adata.X[:5, :5]
        if np.all(first_values >= 0) and np.all(np.isclose(first_values, first_values.astype(int))):
            adata.obs['n_counts'] = adata.X.sum(axis=1).A1 if issparse(adata.X) else adata.X.sum(axis=1)
            adata.obs['n_genes'] = np.array((adata.X > 0).sum(axis=1)).flatten() if issparse(adata.X) else (adata.X > 0).sum(axis=1)

    # 2. Setup Plotting Data
    sample_order = adata.obs[sample_column].unique()
    cluster_colors = adata.uns.get(f'{color_column}_colors', None) if color_column else None

    # 3. Define Grid Logic
    # Dynamically set rows: 2 for boxplots, 1 for barplot
    rows_needed = []
    if boxplots: rows_needed.extend([1, 1]) # UMI and Gene rows
    if barplot: rows_needed.append(2)        # Barplot row (weighted higher)
    
    if not rows_needed:
        raise ValueError("At least one of 'barplot' or 'boxplots' must be True.")

    fig, axs = plt.subplots(
        len(rows_needed), 1, 
        figsize=(24, 4 * len(rows_needed)), 
        sharex=True, 
        gridspec_kw={'height_ratios': rows_needed}
    )
    
    # Handle axs indexing if only one plot exists
    if len(rows_needed) == 1:
        axs = [axs]
    
    ax_idx = 0

    # 4. Boxplots (Only if requested)
    if boxplots:
        # UMI Plot
        umi_data = pd.DataFrame({'Sample': np.repeat(sample_order, [len(adata.obs[adata.obs[sample_column]==s]) for s in sample_order]),
                                 'UMIs': adata.obs.groupby(sample_column)["n_counts"].apply(list).loc[sample_order].explode()})
        sns.boxplot(data=umi_data, x="Sample", y="UMIs", ax=axs[ax_idx], color="white", fliersize=1, linewidth=1)
        axs[ax_idx].set_ylabel("UMIs per cell")
        ax_idx += 1

        # Gene Plot
        gene_data = pd.DataFrame({'Sample': np.repeat(sample_order, [len(adata.obs[adata.obs[sample_column]==s]) for s in sample_order]),
                                  'Genes': adata.obs.groupby(sample_column)["n_genes"].apply(list).loc[sample_order].explode()})
        sns.boxplot(data=gene_data, x="Sample", y="Genes", ax=axs[ax_idx], color="white", fliersize=1, linewidth=1)
        axs[ax_idx].set_ylabel("Genes per cell")
        ax_idx += 1

    # 5. Barplot (Only if requested)
    if barplot:
        proportions = adata.obs.groupby([sample_column, color_column]).size().unstack(fill_value=0)
        proportions = proportions.div(proportions.sum(axis=1), axis=0).reindex(sample_order)
        
        proportions.plot(kind="bar", stacked=True, ax=axs[ax_idx], width=1, legend=False, color=cluster_colors)
        axs[ax_idx].set_ylabel("Cell class proportions")
        axs[ax_idx].set_xlabel("Sample")
        
        if cluster_colors is not None:
            axs[ax_idx].legend(bbox_to_anchor=(1.01, 1), loc="upper left", title=color_column)
            
    # Clean up X-axis labels for the bottom-most plot
    axs[-1].tick_params(axis='x', rotation=90, labelsize=7)

    plt.tight_layout()
    return fig
    
fig = plot_stacked_figure(
    adata, 
    sample_column="sample", 
    color_column="cell_type", 
    barplot=True, 
    boxplots=False, 
    recalculate_columns=True
)

# 2. Save PDF
fig.savefig(
    tbl_dir + "pct_cells_per_smpl.pdf",
    dpi=300,
    bbox_inches="tight"
)

# 3. Save SVG
fig.savefig(
    tbl_dir + "pct_cells_per_smpl.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)